In [ ]:
!pip -q install --upgrade pandas numpy matplotlib seaborn transformers


In [ ]:
import json
import warnings
from dataclasses import dataclass, field
from typing import Optional, Dict, Any, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings("ignore")
sns.set(style="whitegrid")
np.random.seed(42)


In [ ]:
MODEL_NAME = "microsoft/Phi-3-mini-4k-instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device).eval()

def llm_generate(messages, max_new_tokens=400, temperature=0.4):
    chat_prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(chat_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=False
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)


In [ ]:
@dataclass
class AnalysisState:
    df: Optional[pd.DataFrame] = None
    df_preview: Optional[pd.DataFrame] = None

    schema_summary: Dict[str, Any] = field(default_factory=dict)
    plot_summaries: List[Dict[str, Any]] = field(default_factory=list)
    text_insights: Dict[str, Any] = field(default_factory=dict)
    ts_insights: Dict[str, Any] = field(default_factory=dict)
    narrative: str = ""
    qa_report: Dict[str, Any] = field(default_factory=dict)

    logs: List[Dict[str, Any]] = field(default_factory=list)

    def log(self, agent: str, message: str, level: str = "INFO"):
        entry = {"agent": agent, "message": message, "level": level}
        print(f"[{agent}] {message}")
        self.logs.append(entry)


In [ ]:
class AgentMessage:
    def __init__(self, sender, receiver, type, payload):
        self.sender = sender
        self.receiver = receiver
        self.type = type
        self.payload = payload

class BaseAgent:
    def __init__(self, name: str):
        self.name = name

    def handle(self, msg: AgentMessage, state: AnalysisState):
        raise NotImplementedError


In [ ]:
class SchemaAgent(BaseAgent):
    def __init__(self):
        super().__init__("SchemaAgent")

    def handle(self, msg, state):
        df = state.df
        state.log(self.name, "Starting schema analysis")

        summary = {
            "rows": len(df),
            "columns": list(df.columns),
            "dtypes": df.dtypes.astype(str).to_dict(),
            "missing_percent": (df.isna().mean() * 100).round(2).to_dict(),
            "n_unique": df.nunique().to_dict(),
            "example_rows": df.head(3).to_dict(orient="records"),
        }

        summary["numeric_columns"] = df.select_dtypes(include=np.number).columns.tolist()
        summary["categorical_columns"] = df.select_dtypes(include=["object", "category"]).columns.tolist()

        # Try datetime parsing
        dt_cols = []
        for col in df.columns:
            if df[col].dtype == "object":
                try:
                    df[col] = pd.to_datetime(df[col], errors="raise", utc=True)
                    dt_cols.append(col)
                except:
                    pass

        summary["datetime_columns"] = dt_cols
        state.schema_summary = summary
        state.df = df

        state.log(self.name, "Schema analysis complete")
        return summary


In [ ]:
class PlotAgent(BaseAgent):
    def __init__(self):
        super().__init__("PlotAgent")

    def handle(self, msg, state):
        df = state.df
        schema = state.schema_summary
        plots = []

        state.log(self.name, "Generating plots")

        # Numeric
        numeric_cols = schema["numeric_columns"]
        if numeric_cols:
            df[numeric_cols].hist(figsize=(15, 10), bins=30)
            plt.show()
            plots.append({"type": "numeric_histograms", "columns": numeric_cols})

        # Categorical
        for col in schema["categorical_columns"][:6]:
            vc = df[col].value_counts().head(20)
            plt.figure(figsize=(10, 4))
            sns.barplot(x=vc.index.astype(str), y=vc.values)
            plt.xticks(rotation=45)
            plt.show()
            plots.append({"type": "value_counts", "column": col})

        # Missing heatmap
        if df.isna().sum().sum() > 0:
            plt.figure(figsize=(10, 5))
            sns.heatmap(df.isna(), cbar=False)
            plt.show()
            plots.append({"type": "missing_heatmap"})

        # Datetime
        dt_cols = schema["datetime_columns"]
        if dt_cols:
            col = dt_cols[0]
            df_sorted = df.sort_values(col)
            counts = df_sorted.groupby(col).size()
            counts.plot(figsize=(12, 5))
            plt.xticks(rotation=45)
            plt.show()
            plots.append({"type": "time_series", "column": col})

        state.plot_summaries = plots
        state.log(self.name, f"Generated {len(plots)} plots")
        return plots


In [ ]:
class TextAgent(BaseAgent):
    def __init__(self):
        super().__init__("TextAgent")

    def handle(self, msg, state):
        df = state.df
        schema = state.schema_summary
        insights = {}

        state.log(self.name, "Analyzing text columns")

        for col in schema["categorical_columns"]:
            avg_len = df[col].astype(str).str.len().mean()
            if avg_len < 20:
                continue

            text = " ".join(df[col].astype(str)).lower()
            import re
            from collections import Counter
            words = re.findall(r"\b\w+\b", text)
            common = Counter(words).most_common(25)

            insights[col] = {"avg_length": avg_len, "top_words": common}

        state.text_insights = insights
        state.log(self.name, f"Text insights for {len(insights)} columns")
        return insights


In [ ]:
class TimeSeriesAgent(BaseAgent):
    def __init__(self):
        super().__init__("TimeSeriesAgent")

    def handle(self, msg, state):
        df = state.df
        dt_cols = state.schema_summary["datetime_columns"]
        insights = {}

        state.log(self.name, "Analyzing time-series")

        if dt_cols:
            col = dt_cols[0]
            df_sorted = df.sort_values(col)
            counts = df_sorted.groupby(col).size()
            insights = {
                "datetime_column": col,
                "n_unique_timestamps": len(counts),
                "example_counts": counts.head(10).to_dict()
            }

        state.ts_insights = insights
        return insights


In [ ]:
class NarrativeAgent(BaseAgent):
    def __init__(self):
        super().__init__("NarrativeAgent")

    def _make_json_safe(self, obj):
        """Recursively convert all non‑JSON‑serializable objects to strings."""
        if isinstance(obj, dict):
            return {str(k): self._make_json_safe(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._make_json_safe(v) for v in obj]
        elif isinstance(obj, (np.integer, np.floating)):
            return obj.item()
        elif isinstance(obj, (pd.Timestamp, pd.Timedelta)):
            return str(obj)
        else:
            try:
                json.dumps(obj)
                return obj
            except:
                return str(obj)

    def handle(self, msg, state):
        # Build structured summary
        summary = {
            "schema": state.schema_summary,
            "plots": state.plot_summaries,
            "text": state.text_insights,
            "time_series": state.ts_insights
        }

        # ✅ Convert everything to JSON‑safe format
        safe_summary = self._make_json_safe(summary)

        prompt = f"""
You are a senior data analyst. Write a grounded EDA summary using ONLY this structured data:
{json.dumps(safe_summary, indent=2)}
"""

        messages = [
            {"role": "system", "content": "You are a precise, grounded analyst."},
            {"role": "user", "content": prompt}
        ]

        narrative = llm_generate(messages, max_new_tokens=450)
        state.narrative = narrative
        state.log(self.name, "Narrative generated")
        return narrative


In [ ]:
class QAAgent(BaseAgent):
    def __init__(self):
        super().__init__("QAAgent")

    def handle(self, msg, state):
        issues = []
        s = state.schema_summary

        if s["numeric_columns"] and "No numeric" in state.narrative:
            issues.append("Narrative incorrectly states no numeric columns.")

        if s["datetime_columns"] and not state.ts_insights:
            issues.append("Datetime exists but no time-series insights.")

        if s["categorical_columns"] and not state.text_insights:
            issues.append("Text columns exist but no text analysis.")

        state.qa_report = {
            "issues_detected": issues,
            "n_issues": len(issues),
            "status": "pass" if len(issues) == 0 else "needs_improvement"
        }

        state.log(self.name, f"QA complete — {len(issues)} issues")
        return state.qa_report


In [ ]:
class OrchestratorAgent(BaseAgent):
    def __init__(self, schema, plot, text, ts, narrative, qa):
        super().__init__("Orchestrator")
        self.schema_agent = schema
        self.plot_agent = plot
        self.text_agent = text
        self.ts_agent = ts
        self.narrative_agent = narrative
        self.qa_agent = qa

    def run(self, df, goal="full_eda"):
        state = AnalysisState()
        state.df = df.copy()
        state.df_preview = df.head()

        state.log(self.name, f"Starting pipeline with goal: {goal}")

        # Always run schema
        self.schema_agent.handle(None, state)

        # Goal‑based branching
        if goal in ["full_eda", "bias_analysis", "text_focus"]:
            self.plot_agent.handle(None, state)

        if goal in ["full_eda", "text_focus"]:
            self.text_agent.handle(None, state)

        if goal in ["full_eda", "time_series_focus"]:
            self.ts_agent.handle(None, state)

        # Narrative
        self.narrative_agent.handle(None, state)

        # QA
        self.qa_agent.handle(None, state)

        state.log(self.name, "Pipeline complete")
        return state


In [ ]:
csv_path = "/content/news_sentiment_analysis.csv"
df = pd.read_csv(csv_path)

schema = SchemaAgent()
plot = PlotAgent()
text = TextAgent()
ts = TimeSeriesAgent()
narrative = NarrativeAgent()
qa = QAAgent()

orchestrator = OrchestratorAgent(schema, plot, text, ts, narrative, qa)

state = orchestrator.run(df, goal="full_eda")

print("\n======================")
print("✅ FINAL NARRATIVE")
print("======================\n")
print(state.narrative)

print("\n======================")
print("✅ QA REPORT")
print("======================\n")
print(state.qa_report)
